In [ ]:
import json
from typing import List
import pandas as pd
import os
import datetime



## Testing

In [ ]:
# Testing

data = json.load(open("../master_data/2024/1415728.json"))
data.keys()

In [ ]:
data['info'].keys()

In [ ]:
# 2 innings (0,1)
data['innings'][0]
data['innings'][0].keys() # dict_keys(['team', 'overs', 'powerplays'])
len(data['innings'][0]['overs']) # 20 overs
data['innings'][0]['overs'][0].keys() # dict_keys(['over','deliveris'])
len(data['innings'][0]['overs'][0]['deliveries']) #7
data['innings'][0]['overs'][0]['deliveries']


In [ ]:
import datetime 
year = datetime.datetime.strptime(data['info']['dates'][0], '%Y-%m-%d').year
print(year)
2016 == year

# Move to year folder

In [ ]:
# filter data by year and send to 2024 folder
import os
import pandas as pd
import json
import shutil

# Get all files in the master_data directory
master_data_dir = '../master_data/Years/'
year = 2014

# Create a new directory for the filtered data
filtered_data_dir = os.path.join(master_data_dir, f'{year}')
print(filtered_data_dir)
os.makedirs(filtered_data_dir, exist_ok=True)

# Filter and save data for the specified year
for file in os.listdir(master_data_dir):
    if file.endswith('.json'):
        file_path = os.path.join(master_data_dir, file)
        data = json.load(open(file_path))
        dataset_year = datetime.datetime.strptime(data['info']['dates'][0], '%Y-%m-%d').year
        if dataset_year == year:
            shutil.copy(file_path, os.path.join(filtered_data_dir, file))




In [ ]:
dates = []
for file in os.listdir(filtered_data_dir):
    if file.endswith('.json'):
        file_path = os.path.join(filtered_data_dir, file)
        data = json.load(open(file_path))
        dates.append(datetime.datetime.strptime(data['info']['dates'][0], '%Y-%m-%d'))









# Create yearly math dataset

In [ ]:
def create_year_wise_comp_match_dataset(path:str) -> List:
    """
    Create a dataset of all the matches in a year
    """

    city,dates,event,event_match_number,group,match_type,match_type_number,outcome,overs,season,teams,venue = [],[],[],[],[],[],[],[],[],[],[],[]   

    for file in os.listdir(path):
        print(file)
        if file.endswith('.json'):
            file_path = os.path.join(path, file)
            data = json.load(open(file_path))
            try:
                city.append(data['info']['city'])
            except:
                city.append("NG")
            dates.append(data['info']['dates'][0])   
            event.append(data['info']['event']['name'])

            try:
                event_match_number.append(data['info']['event']['match_number'])
                group.append(data['info']['event']['group'])
            except:
                event_match_number.append(data['info']['event']['stage'])
                group.append("Not Applicable")

            match_type.append(data['info']['match_type'])
            outcome.append(data['info']['outcome'])
            overs.append(data['info']['overs'])
            season.append(data['info']['season'])
            teams.append(data['info']['teams'])
            venue.append(data['info']['venue'])
    
    return match_type,event,dates,city,event_match_number,group,overs,season,teams,venue,outcome







In [ ]:
json_path = "../master_data/Years/2016/"
match_type,event,dates,city,event_match_number,group,overs,season,teams,venue,outcome = create_year_wise_comp_match_dataset(json_path)

# Master dataset from all JSON

**JSON pattern (Cricsheet v1.0.0 / 1.1.0):**
- **Root:** `meta`, `info`, `innings`
- **info:** match_id (from filename), dates, city, venue, teams, toss (winner, decision), outcome (winner, by.runs or by.wickets), event (name, match_number, group, stage?), season, player_of_match, players, overs, balls_per_over
- **innings:** array of `{ team, overs: [ { over, deliveries: [ { batter, bowler, non_striker, runs, extras?, wickets? } ] } ], powerplays?, target? }`
- **delivery:** runs (batter, extras, total); optional: extras (wides, noballs…), wickets (player_out, kind, fielders)

Output: **matches** (one row per match) + **deliveries** (one row per ball), same columns across all years.

In [11]:
import os 
import json

MASTER_JSON_DIR = os.path.join("..", "master_data", "Years")
OUTPUT_DIR = os.path.join("..","master_data", "dataset")

from typing import Tuple, List
import pandas as pd

def get_all_json_paths(base_dir:str) -> List[Tuple[str, str, str]]:
    """Collect all match JSON paths: (year, match_id, file_path)."""
    paths = []
    for year_folder in sorted(os.listdir(base_dir)):
        year_path = os.path.join(base_dir, year_folder)
        if not os.path.isdir(year_path):
            continue
        for f in os.listdir(year_path):
            if f.endswith(".json"):
                match_id = os.path.splitext(f)[0]
                paths.append((year_folder, match_id, os.path.join(year_path, f)))
    return paths


In [ ]:
# Paths: notebook is in scource/, data in master_data/Years/{year}/{match_id}.json


def parse_match_info(data, match_id):
    """Extract one row of match-level info (same format for all years)."""
    
    info = data.get("info", {})
    outcome = info.get("outcome", {})
    by = outcome.get("by", {}) or {}
    event = info.get("event", {}) or {}
    toss = info.get("toss", {}) or {}
    teams = info.get("teams", [])
    date_str = info.get("dates", [None])[0]
    return {
        "match_id": match_id,
        "date": date_str,
        "season": info.get("season"),
        "city": info.get("city"),
        "venue": info.get("venue"),
        "team1": teams[0] if len(teams) > 0 else None,
        "team2": teams[1] if len(teams) > 1 else None,
        "toss_winner": toss.get("winner"),
        "toss_decision": toss.get("decision"),
        "winner": outcome.get("winner"),
        "result_type": "runs" if "runs" in by else ("wickets" if "wickets" in by else None),
        "result_margin": by.get("runs") or by.get("wickets"),
        "event_name": event.get("name"),
        "event_match_number": event.get("match_number"),
        "event_group": event.get("group"),
        "event_stage": event.get("stage"),
        "player_of_match": (info.get("player_of_match") or [None])[0],
        "overs": info.get("overs"),
        "balls_per_over": info.get("balls_per_over"),
        "gender": info.get("gender"),
        "match_type": info.get("match_type"),
        "data_version": data.get("meta", {}).get("data_version"),
    }


def parse_deliveries(data, match_id):
    """Flatten ball-by-ball into rows (same format for all years)."""
    rows = []
    innings_list = data.get("innings", [])
    teams = data.get("info", {}).get("teams", [])
    for inn_no, inn in enumerate(innings_list, start=1):
        batting_team = inn.get("team")
        bowling_team = teams[1 - teams.index(batting_team)] if batting_team in teams and len(teams) == 2 else None
        for over_block in inn.get("overs", []):
            over_num = over_block.get("over", 0)
            for ball_idx, d in enumerate(over_block.get("deliveries", []), start=1):
                runs = d.get("runs", {})
                wickets = d.get("wickets", [])
                w = wickets[0] if wickets else None
                fielder = None
                if w and w.get("fielders"):
                    fielder = w["fielders"][0].get("name")
                extras = d.get("extras", {})
                extras_type = ",".join(extras.keys()) if extras else None
                rows.append({
                    "match_id": match_id,
                    "inning_no": inn_no,
                    "batting_team": batting_team,
                    "bowling_team": bowling_team,
                    "over": over_num,
                    "ball_in_over": ball_idx,
                    "batter": d.get("batter"),
                    "bowler": d.get("bowler"),
                    "non_striker": d.get("non_striker"),
                    "runs_batter": runs.get("batter", 0),
                    "runs_extras": runs.get("extras", 0),
                    "runs_total": runs.get("total", 0),
                    "is_wicket": 1 if w else 0,
                    "wicket_player_out": w.get("player_out") if w else None,
                    "wicket_kind": w.get("kind") if w else None,
                    "wicket_fielder": fielder,
                    "extras_type": extras_type if extras_type else None,
                })
    return rows

def build_master_datasets(base_dir=MASTER_JSON_DIR):
    """Build df_matches and df_deliveries from all JSON files."""
    paths = get_all_json_paths(base_dir)
    match_rows = []
    delivery_rows = []
    for year, match_id, file_path in paths:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"Skip {file_path}: {e}")
            continue
        match_rows.append(parse_match_info(data, match_id))
        delivery_rows.extend(parse_deliveries(data, match_id))
    df_matches = pd.DataFrame(match_rows)
    df_deliveries = pd.DataFrame(delivery_rows)
    return df_matches, df_deliveries

# Build and save
df_matches, df_deliveries = build_master_datasets()
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_matches.to_csv(os.path.join(OUTPUT_DIR, "master_matches.csv"), index=False)
df_deliveries.to_csv(os.path.join(OUTPUT_DIR, "master_deliveries.csv"), index=False)
print("Matches:", df_matches.shape)
print("Deliveries:", df_deliveries.shape)


In [37]:
df_deliveries

,match_id,inning_no,batting_team,bowling_team,over,ball_in_over,batter,bowler,non_striker,runs_batter,runs_extras,runs_total,is_wicket,wicket_player_out,wicket_kind,wicket_fielder,extras_type
0,951309,1,Bangladesh,Netherlands,0,1,Tamim Iqbal,Mudassar Bukhari,Soumya Sarkar,1,0,1,0,None,None,None,None
1,951309,1,Bangladesh,Netherlands,0,2,Soumya Sarkar,Mudassar Bukhari,Tamim Iqbal,0,0,0,0,None,None,None,None
2,951309,1,Bangladesh,Netherlands,0,3,Soumya Sarkar,Mudassar Bukhari,Tamim Iqbal,0,0,0,0,None,None,None,None
3,951309,1,Bangladesh,Netherlands,0,4,Soumya Sarkar,Mudassar Bukhari,Tamim Iqbal,2,0,2,0,None,None,None,None
4,951309,1,Bangladesh,Netherlands,0,5,Soumya Sarkar,Mudassar Bukhari,Tamim Iqbal,4,0,4,0,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33657,1415755,2,South Africa,India,19,3,K Rabada,HH Pandya,KA Maharaj,0,1,1,0,None,None,None,byes
33658,1415755,2,South Africa,India,19,4,KA Maharaj,HH Pandya,K Rabada,0,1,1,0,None,None,None,legbyes
33659,1415755,2,South Africa,India,19,5,K Rabada,HH Pandya,KA Maharaj,0,1,1,0,None,None,None,wides
33660,1415755,2,South Africa,India,19,6,K Rabada,HH Pandya,KA Maharaj,0,0,0,1,K Rabada,caught,SA Yadav,None


In [39]:
# Column names and sample deliveries
print("Matches columns:", list(df_matches.columns))
print("\nDeliveries columns:", list(df_deliveries.columns))
df_deliveries.head(5)

Matches columns: ['match_id', 'date', 'season', 'city', 'venue', 'team1', 'team2', 'toss_winner', 'toss_decision', 'winner', 'result_type', 'result_margin', 'event_name', 'event_match_number', 'event_group', 'event_stage', 'player_of_match', 'overs', 'balls_per_over', 'gender', 'match_type', 'data_version']

Deliveries columns: ['match_id', 'inning_no', 'batting_team', 'bowling_team', 'over', 'ball_in_over', 'batter', 'bowler', 'non_striker', 'runs_batter', 'runs_extras', 'runs_total', 'is_wicket', 'wicket_player_out', 'wicket_kind', 'wicket_fielder', 'extras_type']


,match_id,inning_no,batting_team,bowling_team,over,ball_in_over,batter,bowler,non_striker,runs_batter,runs_extras,runs_total,is_wicket,wicket_player_out,wicket_kind,wicket_fielder,extras_type
0,951309,1,Bangladesh,Netherlands,0,1,Tamim Iqbal,Mudassar Bukhari,Soumya Sarkar,1,0,1,0,None,None,None,None
1,951309,1,Bangladesh,Netherlands,0,2,Soumya Sarkar,Mudassar Bukhari,Tamim Iqbal,0,0,0,0,None,None,None,None
2,951309,1,Bangladesh,Netherlands,0,3,Soumya Sarkar,Mudassar Bukhari,Tamim Iqbal,0,0,0,0,None,None,None,None
3,951309,1,Bangladesh,Netherlands,0,4,Soumya Sarkar,Mudassar Bukhari,Tamim Iqbal,2,0,2,0,None,None,None,None
4,951309,1,Bangladesh,Netherlands,0,5,Soumya Sarkar,Mudassar Bukhari,Tamim Iqbal,4,0,4,0,None,None,None,None


In [82]:
# Commentary file: 2024 matches only. Sheets named "Team A vs Team B_Text_Comm" with col0=over.ball, col2=short, col3=full text.
COMMENTARY_PATH = os.path.join("..", "dataset", "ICC_Mens_T20World_Cup_Data_Text_Commentary.xlsx")

# Normalize team names from commentary sheet to match our JSON (e.g. U.S.A. -> United States of America)
TEAM_ALIASES = {
    "u.s.a.": "United States of America", "usa": "United States of America",
    "india": "India", "ireland": "Ireland", "pakistan": "Pakistan",
    "afghanistan": "Afghanistan", "bangladesh": "Bangladesh", "england": "England",
    "australia": "Australia", "south africa": "South Africa",
}

def normalize_team(t):
    if pd.isna(t) or not str(t).strip():
        return None
    key = str(t).strip().lower()
    return TEAM_ALIASES.get(key, str(t).strip())

def sheet_name_to_teams(sheet_name):
    """e.g. 'Ireland vs India_Text_Comm' -> ('Ireland', 'India')"""
    name = sheet_name.replace("_Text_Comm", "").strip()
    if " vs " in name:
        a, b = name.split(" vs ", 1)
        return (normalize_team(a), normalize_team(b))
    return (None, None)

def match_id_from_teams(team1, team2, matches_2024):
    """Find match_id where teams match (order-independent)."""
    s = {team1, team2}
    for _, r in matches_2024.iterrows():
        if {r["team1"], r["team2"]} == s:
            return str(r["match_id"])
    return None

def parse_over_ball(val):
    """e.g. '0.1', '1.5' -> (0, 1), (1, 5). Returns None if not over.ball."""
    if pd.isna(val):
        return None
    s = str(val).strip()
    if not s or "." not in s:
        return None
    parts = s.split(".", 1)
    try:
        over = int(parts[0])
        ball = int(parts[1])
        return (over, ball)
    except ValueError:
        return None

def load_commentary_per_sheet(path, sheet_name, matches_2024):
    """Load one commentary sheet; return list of (match_id, inning_no, over, ball_in_over, short_comm, commentary)."""
    df = pd.read_excel(path, sheet_name=sheet_name, header=None)
    team1, team2 = sheet_name_to_teams(sheet_name)
    match_id = match_id_from_teams(team1, team2, matches_2024)
    if match_id is None:
        return []
    # Detect innings: "Inning- I" / "Inning- II" or "Inning- 1" / "Inning- 2"
    current_inning = 1
    rows = []
    for i in range(len(df)):
        c0 = df.iloc[i, 0]
        if pd.notna(c0) and isinstance(c0, str):
            lower = c0.lower()
            if "inning" in lower and (" ii " in lower or " 2 " in lower or "2 " in lower):
                current_inning = 2
            elif "inning" in lower and (" i " in lower or " 1 " in lower or "1 " in lower):
                current_inning = 1
        ob = parse_over_ball(c0)
        if ob is None:
            continue
        over_num, ball_num = ob
        short = df.iloc[i, 2] if df.shape[1] > 2 else None
        full = df.iloc[i, 3] if df.shape[1] > 3 else None
        if pd.isna(full):
            full = short
        rows.append((match_id, current_inning, over_num, ball_num, short, full))
    return rows

def build_commentary_df(path, matches_2024):
    """Build DataFrame of commentary rows from all _Text_Comm sheets."""
    xl = pd.ExcelFile(path)
    all_rows = []
    for name in xl.sheet_names:
        if "_Text_Comm" not in name:
            continue
        rows = load_commentary_per_sheet(path, name, matches_2024)
        all_rows.extend(rows)
    if not all_rows:
        return pd.DataFrame(columns=["match_id", "inning_no", "over", "ball_in_over", "commentary_short", "commentary"])
    return pd.DataFrame(all_rows, columns=["match_id", "inning_no", "over", "ball_in_over", "commentary_short", "commentary"])

# Filter 2024 matches (by date)
if not pd.api.types.is_datetime64_any_dtype(df_matches["date"]):
    df_matches["date"] = pd.to_datetime(df_matches["date"], errors="coerce")
matches_2024 = df_matches[df_matches["date"].dt.year == 2024].copy()
df_comm = build_commentary_df(COMMENTARY_PATH, matches_2024)
# Ensure match_id is string for consistent merge
df_deliveries["match_id"] = df_deliveries["match_id"].astype(str)
if not df_comm.empty:
    df_comm["match_id"] = df_comm["match_id"].astype(str)
print("Commentary rows loaded:", len(df_comm))
if len(df_comm) > 0:
    print("Matches in commentary:", df_comm["match_id"].nunique())

# Merge: align by (match_id, inning_no, over, ball_in_over) using sequence (commentary can have duplicate over.ball for wides)
# So we assign by order within (match_id, inning_no, over).
def add_commentary_to_deliveries(deliveries, commentary_df):
    if commentary_df.empty:
        deliveries["commentary_short"] = None
        deliveries["commentary"] = None
        return deliveries
    d = deliveries.copy()
    d["commentary_short"] = None
    d["commentary"] = None
    d["_order"] = range(len(d))
    for (mid, inn), g_d in d.groupby(["match_id", "inning_no"]):
        g_d = g_d.sort_values(["over", "ball_in_over"])
        g_c = commentary_df[(commentary_df["match_id"] == mid) & (commentary_df["inning_no"] == inn)].sort_values(["over", "ball_in_over"])
        if g_c.empty:
            continue
        idx_d = g_d.index.tolist()
        idx_c = g_c.index.tolist()
        for i, didx in enumerate(idx_d):
            if i < len(idx_c):
                d.loc[didx, "commentary_short"] = g_c.loc[idx_c[i], "commentary_short"]
                d.loc[didx, "commentary"] = g_c.loc[idx_c[i], "commentary"]
    d = d.drop(columns=["_order"])
    return d

df_deliveries_with_comm = add_commentary_to_deliveries(df_deliveries, df_comm)
# Save enriched deliveries
out_path = os.path.join(OUTPUT_DIR, "master_deliveries_with_commentary.csv")
df_deliveries_with_comm.to_csv(out_path, index=False)
print("Saved:", out_path)
# Show sample where commentary is present
sample = df_deliveries_with_comm[df_deliveries_with_comm["commentary"].notna()].head(5)
sample[["match_id", "inning_no", "over", "ball_in_over", "batter", "bowler", "runs_total", "commentary_short", "commentary"]]

Commentary rows loaded: 667
Matches in commentary: 4
Saved: ..\master_data\dataset\master_deliveries_with_commentary.csv


,match_id,inning_no,over,ball_in_over,batter,bowler,runs_total,commentary_short,commentary
25191,1415708,1,0,1,A Balbirnie,Arshdeep Singh,1,"Arshdeep to Balbirnie, 1 run","full and swinging back into middle and leg, ne..."
25192,1415708,1,0,2,PR Stirling,Arshdeep Singh,2,"Adair to Rohit Sharma, 1 wide",back of a length and shaping away wide of off ...
25193,1415708,1,0,3,PR Stirling,Arshdeep Singh,0,"Adair to Rohit Sharma, no run","shortish, 133kph outside off, dabbed down towa..."
25194,1415708,1,0,4,PR Stirling,Arshdeep Singh,0,"Arshdeep to Stirling, 2 runs",gives him the charge and punches this length b...
25195,1415708,1,0,5,PR Stirling,Arshdeep Singh,0,"Adair to Rohit Sharma, no run","length ball, 131kph aiming for the top of off...."


## Commentary add-on (2024)

Load **ICC_Mens_T20World_Cup_Data_Text_Commentary.xlsx**: 
each sheet = match (e.g. Ireland vs India_Text_Comm) 
with col0 = over.ball, col2/col3 = short/full text. 
Match to deliveries by (match_id, inning_no, over, ball_in_over). 
Output: **master_deliveries_with_commentary.csv**.

In [84]:
df_deliveries_with_comm[df_deliveries_with_comm["commentary"].notna()]

,match_id,inning_no,batting_team,bowling_team,over,ball_in_over,batter,bowler,non_striker,runs_batter,runs_extras,runs_total,is_wicket,wicket_player_out,wicket_kind,wicket_fielder,extras_type,commentary_short,commentary
25191,1415708,1,Ireland,India,0,1,A Balbirnie,Arshdeep Singh,PR Stirling,1,0,1,0,None,None,None,None,"Arshdeep to Balbirnie, 1 run","full and swinging back into middle and leg, ne..."
25192,1415708,1,Ireland,India,0,2,PR Stirling,Arshdeep Singh,A Balbirnie,2,0,2,0,None,None,None,None,"Adair to Rohit Sharma, 1 wide",back of a length and shaping away wide of off ...
25193,1415708,1,Ireland,India,0,3,PR Stirling,Arshdeep Singh,A Balbirnie,0,0,0,0,None,None,None,None,"Adair to Rohit Sharma, no run","shortish, 133kph outside off, dabbed down towa..."
25194,1415708,1,Ireland,India,0,4,PR Stirling,Arshdeep Singh,A Balbirnie,0,0,0,0,None,None,None,None,"Arshdeep to Stirling, 2 runs",gives him the charge and punches this length b...
25195,1415708,1,Ireland,India,0,5,PR Stirling,Arshdeep Singh,A Balbirnie,0,0,0,0,None,None,None,None,"Adair to Rohit Sharma, no run","length ball, 131kph aiming for the top of off...."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32289,1415747,1,India,Bangladesh,0,2,RG Sharma,Mahedi Hasan,V Kohli,0,0,0,0,None,None,None,None,"Mahedi Hasan to Rohit Sharma, no run","quicker length ball on off stump, clipped towa..."
32290,1415747,1,India,Bangladesh,0,3,RG Sharma,Mahedi Hasan,V Kohli,1,0,1,0,None,None,None,None,"Mahedi Hasan to Rohit Sharma, 1 run","flat, fired in to finish around leg stump. Mak..."
32291,1415747,1,India,Bangladesh,0,4,V Kohli,Mahedi Hasan,RG Sharma,0,0,0,0,None,None,None,None,"Mahedi Hasan to Kohli, no run","tossed up on middle and leg, gets forward for ..."
32292,1415747,1,India,Bangladesh,0,5,V Kohli,Mahedi Hasan,RG Sharma,1,0,1,0,None,None,None,None,"Mahedi Hasan to Kohli, 1 run","length ball turning towards leg stump, and thi..."


# Team & player info dataset

From each match JSON: **team**, **player_name**, **player_id** (registry), **primary_role** (batter / bowler / allrounder from delivery counts), **is_player_of_match**, plus **match_id**, **date**, **season**, **event_name**. Saved as **master_data/dataset/team_player_info.csv**.

In [12]:
def get_role_from_innings(data, match_id):
    """From innings, count balls as batter (batter or non_striker) and as bowler per player. Return dict player_name -> 'batter'|'bowler'|'allrounder'."""
    batted = {}
    bowled = {}
    for inn in data.get("innings", []):
        for over_block in inn.get("overs", []):
            for d in over_block.get("deliveries", []):
                b = d.get("batter")
                ns = d.get("non_striker")
                bowler = d.get("bowler")
                if b:
                    batted[b] = batted.get(b, 0) + 1
                if ns:
                    batted[ns] = batted.get(ns, 0) + 1
                if bowler:
                    bowled[bowler] = bowled.get(bowler, 0) + 1
    role = {}
    all_players = set(batted) | set(bowled)
    for p in all_players:
        nb, nbowl = batted.get(p, 0), bowled.get(p, 0)
        if nb == 0:
            role[p] = "bowler"
        elif nbowl == 0:
            role[p] = "batter"
        else:
            role[p] = "allrounder"
    return role

def build_team_player_info(base_dir=MASTER_JSON_DIR):
    """Build one row per (match_id, team, player_name) with team, player, role, match info."""
    paths = get_all_json_paths(base_dir)
    rows = []
    for year, match_id, file_path in paths:
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except Exception as e:
            print(f"Skip {file_path}: {e}")
            continue
        info = data.get("info", {})
        players_by_team = info.get("players", {})
        registry = (info.get("registry") or {}).get("people", {})
        date_str = (info.get("dates") or [None])[0]
        season = info.get("season")
        event = info.get("event") or {}
        event_name = event.get("name")
        player_of_match = (info.get("player_of_match") or [None])[0]
        roles = get_role_from_innings(data, match_id)
        for team, names in players_by_team.items():
            for player_name in names:
                player_id = registry.get(player_name)
                primary_role = roles.get(player_name, "batter")
                is_pom = 1 if player_name == player_of_match else 0
                rows.append({
                    "match_id": str(match_id),
                    "date": date_str,
                    "season": season,
                    "event_name": event_name,
                    "team": team,
                    "player_name": player_name,
                    "player_id": player_id,
                    "primary_role": primary_role,
                    "is_player_of_match": is_pom,
                })
    return pd.DataFrame(rows)

df_team_player = build_team_player_info()
os.makedirs(OUTPUT_DIR, exist_ok=True)
out_path = os.path.join(OUTPUT_DIR, "team_player_info.csv")
df_team_player.to_csv(out_path, index=False)
print("Team/player info shape:", df_team_player.shape)
print("Saved:", out_path)
df_team_player.head(10)

Team/player info shape: (3279, 9)
Saved: ..\master_data\dataset\team_player_info.csv


,match_id,date,season,event_name,team,player_name,player_id,primary_role,is_player_of_match
0,951309,2016-03-09,2015/16,World T20,Bangladesh,Tamim Iqbal,3b041a12,batter,1
1,951309,2016-03-09,2015/16,World T20,Bangladesh,Soumya Sarkar,4d9f9686,batter,0
2,951309,2016-03-09,2015/16,World T20,Bangladesh,Sabbir Rahman,7147f314,batter,0
3,951309,2016-03-09,2015/16,World T20,Bangladesh,Shakib Al Hasan,7dc35884,allrounder,0
4,951309,2016-03-09,2015/16,World T20,Bangladesh,Mahmudullah,bf1d7d3e,allrounder,0
5,951309,2016-03-09,2015/16,World T20,Bangladesh,Mushfiqur Rahim,a94e08ea,batter,0
6,951309,2016-03-09,2015/16,World T20,Bangladesh,Nasir Hossain,6eea0b32,allrounder,0
7,951309,2016-03-09,2015/16,World T20,Bangladesh,Mashrafe Mortaza,e186f49c,allrounder,0
8,951309,2016-03-09,2015/16,World T20,Bangladesh,Arafat Sunny,1f7adc8a,allrounder,0
9,951309,2016-03-09,2015/16,World T20,Bangladesh,Taskin Ahmed,ef18b66e,bowler,0


In [13]:
df_team_player

,match_id,date,season,event_name,team,player_name,player_id,primary_role,is_player_of_match
0,951309,2016-03-09,2015/16,World T20,Bangladesh,Tamim Iqbal,3b041a12,batter,1
1,951309,2016-03-09,2015/16,World T20,Bangladesh,Soumya Sarkar,4d9f9686,batter,0
2,951309,2016-03-09,2015/16,World T20,Bangladesh,Sabbir Rahman,7147f314,batter,0
3,951309,2016-03-09,2015/16,World T20,Bangladesh,Shakib Al Hasan,7dc35884,allrounder,0
4,951309,2016-03-09,2015/16,World T20,Bangladesh,Mahmudullah,bf1d7d3e,allrounder,0
...,...,...,...,...,...,...,...,...,...
3274,1415755,2024-06-29,2024,ICC Men's T20 World Cup,South Africa,M Jansen,81c36ee9,allrounder,0
3275,1415755,2024-06-29,2024,ICC Men's T20 World Cup,South Africa,KA Maharaj,0b60eb09,allrounder,0
3276,1415755,2024-06-29,2024,ICC Men's T20 World Cup,South Africa,K Rabada,e62dd25d,allrounder,0
3277,1415755,2024-06-29,2024,ICC Men's T20 World Cup,South Africa,A Nortje,acdc62f5,allrounder,0


# Player statistics (universal standard)

From **df_deliveries** and **df_matches**: batting stats (matches, innings, runs, balls, average, strike rate, 50s, 100s, 4s, 6s) and bowling stats (matches, overs, runs, wickets, average, economy, strike rate, 4w, 5w). Saves **batting_stats.csv**, **bowling_stats.csv**, and **player_stats.csv** (one row per player, both batting and bowling columns) to **master_data/dataset**.

In [15]:
import numpy as np

# Ensure we have df_deliveries (run master dataset cell first)
df_deliveries = pd.read_csv("../master_data/dataset/master_deliveries.csv")
dd = df_deliveries.copy()
dd["match_id"] = dd["match_id"].astype(str)

# --- Batting stats (universal standard) ---
bat = dd[dd["batter"].notna()].copy()
# Balls faced: exclude wides (batter didn't face)
bat["balls_faced"] = 1
bat.loc[bat["extras_type"].fillna("").str.contains("wides", case=False, na=False), "balls_faced"] = 0
bat_agg = bat.groupby("batter").agg(
    runs=("runs_batter", "sum"),
    balls_faced=("balls_faced", "sum"),
).reset_index()
bat_agg.rename(columns={"batter": "player_name"}, inplace=True)
# Innings and dismissals
innings_batted = bat.groupby(["batter", "match_id", "inning_no"]).agg(balls=("balls_faced", "sum"), runs=("runs_batter", "sum")).reset_index()
dismissals = dd[dd["wicket_player_out"].notna()].groupby("wicket_player_out").size().reset_index(name="dismissals")
dismissals.rename(columns={"wicket_player_out": "player_name"}, inplace=True)
innings_count = innings_batted.groupby("batter").size().reset_index(name="innings")
innings_count.rename(columns={"batter": "player_name"}, inplace=True)
matches_bat = innings_batted.groupby("batter")["match_id"].nunique().reset_index(name="matches")
matches_bat.rename(columns={"batter": "player_name"}, inplace=True)
# 50s and 100s (runs per innings)
innings_runs = innings_batted.groupby(["batter", "match_id", "inning_no"])["runs"].sum().reset_index()
fifties = innings_runs[innings_runs["runs"] >= 50].groupby("batter").size().reset_index(name="fifties")
fifties.rename(columns={"batter": "player_name"}, inplace=True)
hundreds = innings_runs[innings_runs["runs"] >= 100].groupby("batter").size().reset_index(name="hundreds")
hundreds.rename(columns={"batter": "player_name"}, inplace=True)
# Fours and sixes
fours = bat[bat["runs_batter"] == 4].groupby("batter").size().reset_index(name="fours")
fours.rename(columns={"batter": "player_name"}, inplace=True)
sixes = bat[bat["runs_batter"] == 6].groupby("batter").size().reset_index(name="sixes")
sixes.rename(columns={"batter": "player_name"}, inplace=True)

df_batting = bat_agg.merge(innings_count, on="player_name", how="left")
df_batting = df_batting.merge(dismissals, on="player_name", how="left")
df_batting = df_batting.merge(matches_bat, on="player_name", how="left")
df_batting = df_batting.merge(fifties, on="player_name", how="left")
df_batting = df_batting.merge(hundreds, on="player_name", how="left")
df_batting = df_batting.merge(fours, on="player_name", how="left")
df_batting = df_batting.merge(sixes, on="player_name", how="left")
df_batting["dismissals"] = df_batting["dismissals"].fillna(0).astype(int)
df_batting["not_outs"] = df_batting["innings"] - df_batting["dismissals"]
df_batting["average"] = np.where(df_batting["dismissals"] > 0, (df_batting["runs"] / df_batting["dismissals"]).round(2), None)
df_batting["strike_rate"] = np.where(df_batting["balls_faced"] > 0, (df_batting["runs"] / df_batting["balls_faced"] * 100).round(2), None)
df_batting["fifties"] = df_batting["fifties"].fillna(0).astype(int)
df_batting["hundreds"] = df_batting["hundreds"].fillna(0).astype(int)
df_batting["fours"] = df_batting["fours"].fillna(0).astype(int)
df_batting["sixes"] = df_batting["sixes"].fillna(0).astype(int)
df_batting = df_batting[["player_name", "matches", "innings", "not_outs", "runs", "balls_faced", "average", "strike_rate", "hundreds", "fifties", "fours", "sixes"]]

# --- Bowling stats (universal standard); run out not credited to bowler ---
bowl = dd[dd["bowler"].notna()].copy()
bowl["wicket_for_bowler"] = np.where((bowl["is_wicket"] == 1) & (bowl["wicket_kind"].fillna("").str.lower() != "run out"), 1, 0)
bowl_agg = bowl.groupby("bowler").agg(
    balls_bowled=("bowler", "size"),
    runs_conceded=("runs_total", "sum"),
    wickets=("wicket_for_bowler", "sum"),
).reset_index()
bowl_agg.rename(columns={"bowler": "player_name"}, inplace=True)
bowl_agg["overs"] = (bowl_agg["balls_bowled"] / 6).round(2)
matches_bowl = bowl.groupby("bowler")["match_id"].nunique().reset_index(name="matches")
matches_bowl.rename(columns={"bowler": "player_name"}, inplace=True)
# 4w and 5w per innings
wickets_per_inn = bowl.groupby(["bowler", "match_id", "inning_no"])["wicket_for_bowler"].sum().reset_index()
four_w = wickets_per_inn[wickets_per_inn["wicket_for_bowler"] >= 4].groupby("bowler").size().reset_index(name="four_wickets")
four_w.rename(columns={"bowler": "player_name"}, inplace=True)
five_w = wickets_per_inn[wickets_per_inn["wicket_for_bowler"] >= 5].groupby("bowler").size().reset_index(name="five_wickets")
five_w.rename(columns={"bowler": "player_name"}, inplace=True)

df_bowling = bowl_agg.merge(matches_bowl, on="player_name", how="left")
df_bowling = df_bowling.merge(four_w, on="player_name", how="left")
df_bowling = df_bowling.merge(five_w, on="player_name", how="left")
df_bowling["four_wickets"] = df_bowling["four_wickets"].fillna(0).astype(int)
df_bowling["five_wickets"] = df_bowling["five_wickets"].fillna(0).astype(int)
df_bowling["average"] = np.where(df_bowling["wickets"] > 0, (df_bowling["runs_conceded"] / df_bowling["wickets"]).round(2), None)
df_bowling["economy"] = np.where(df_bowling["balls_bowled"] > 0, (df_bowling["runs_conceded"] / (df_bowling["balls_bowled"] / 6)).round(2), None)
df_bowling["strike_rate"] = np.where(df_bowling["wickets"] > 0, (df_bowling["balls_bowled"] / df_bowling["wickets"]).round(2), None)
df_bowling = df_bowling[["player_name", "matches", "overs", "balls_bowled", "runs_conceded", "wickets", "average", "economy", "strike_rate", "four_wickets", "five_wickets"]]

# --- Save ---
os.makedirs(OUTPUT_DIR, exist_ok=True)
df_batting.to_csv(os.path.join(OUTPUT_DIR, "batting_stats.csv"), index=False)
df_bowling.to_csv(os.path.join(OUTPUT_DIR, "bowling_stats.csv"), index=False)
# Combined: one row per player (batting + bowling columns with _bat / _bowl suffix where overlap)
df_bat_suffix = df_batting.rename(columns={"matches": "matches_bat", "average": "bat_avg", "strike_rate": "bat_sr"})
df_bowl_suffix = df_bowling.rename(columns={"matches": "matches_bowl", "average": "bowl_avg", "strike_rate": "bowl_sr"})
df_player_stats = df_bat_suffix.merge(df_bowl_suffix, on="player_name", how="outer")
df_player_stats.to_csv(os.path.join(OUTPUT_DIR, "player_stats.csv"), index=False)
print("Batting stats:", df_batting.shape)
print("Bowling stats:", df_bowling.shape)
print("Player stats (combined):", df_player_stats.shape)
print("Saved: batting_stats.csv, bowling_stats.csv, player_stats.csv in", OUTPUT_DIR)
df_batting.head(8)

Batting stats: (450, 12)
Bowling stats: (308, 11)
Player stats (combined): (490, 22)
Saved: batting_stats.csv, bowling_stats.csv, player_stats.csv in ..\master_data\dataset


,player_name,matches,innings,not_outs,runs,balls_faced,average,strike_rate,hundreds,fifties,fours,sixes
0,A Balbirnie,13,13,0,245,228,18.85,107.46,0,1,18,11
1,A Bohara,2,2,0,0,2,0.0,0.0,0,0,0,0
2,A Dutt,2,2,1,25,20,25.0,125.0,0,0,2,1
3,A Johnson,3,3,0,89,73,29.67,121.92,0,1,12,4
4,A Nao,2,2,-1,8,13,2.67,61.54,0,0,0,0
5,A Nehra,1,1,0,0,4,0.0,0.0,0,0,0,0
6,A Nortje,4,4,2,8,10,4.0,80.0,0,0,1,0
7,A Sharafu,1,1,0,4,4,4.0,100.0,0,0,0,0


In [16]:
df_batting

,player_name,matches,innings,not_outs,runs,balls_faced,average,strike_rate,hundreds,fifties,fours,sixes
0,A Balbirnie,13,13,0,245,228,18.85,107.46,0,1,18,11
1,A Bohara,2,2,0,0,2,0.0,0.0,0,0,0,0
2,A Dutt,2,2,1,25,20,25.0,125.0,0,0,2,1
3,A Johnson,3,3,0,89,73,29.67,121.92,0,1,12,4
4,A Nao,2,2,-1,8,13,2.67,61.54,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
445,Yuvraj Singh,4,4,0,52,52,13.0,100.0,0,0,3,2
446,ZE Green,10,10,0,110,134,11.0,82.09,0,0,7,2
447,Zahoor Khan,1,1,1,1,1,None,100.0,0,0,0,0
448,Zawar Farid,1,1,0,2,4,2.0,50.0,0,0,0,0


In [17]:
df_bowling

,player_name,matches,overs,balls_bowled,runs_conceded,wickets,average,economy,strike_rate,four_wickets,five_wickets
0,A Bohara,3,8.00,48,61,1,61.0,7.62,48.0,0,0
1,A Dutt,2,7.33,44,41,3,13.67,5.59,14.67,0,0
2,A Nao,3,9.50,57,47,3,15.67,4.95,19.0,0,0
3,A Nehra,5,19.00,114,117,5,23.4,6.16,22.8,0,0
4,A Nortje,18,71.83,431,423,33,12.82,5.89,13.06,3,0
...,...,...,...,...,...,...,...,...,...,...,...
303,Waseem Muhammad,1,2.00,12,16,1,16.0,8.0,12.0,0,0
304,Yuvraj Singh,1,3.50,21,19,1,19.0,5.43,21.0,0,0
305,Zahoor Khan,3,12.67,76,59,5,11.8,4.66,15.2,0,0
306,Zawar Farid,1,2.83,17,25,0,None,8.82,None,0,0


In [18]:
df_player_stats

,player_name,matches_bat,innings,not_outs,runs,balls_faced,bat_avg,bat_sr,hundreds,fifties,...,matches_bowl,overs,balls_bowled,runs_conceded,wickets,bowl_avg,economy,bowl_sr,four_wickets,five_wickets
0,A Balbirnie,13.0,13.0,0.0,245.0,228.0,18.85,107.46,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A Bohara,2.0,2.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,3.0,8.00,48.0,61.0,1.0,61.0,7.62,48.0,0.0,0.0
2,A Dutt,2.0,2.0,1.0,25.0,20.0,25.0,125.0,0.0,0.0,...,2.0,7.33,44.0,41.0,3.0,13.67,5.59,14.67,0.0,0.0
3,A Johnson,3.0,3.0,0.0,89.0,73.0,29.67,121.92,0.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A Nao,2.0,2.0,-1.0,8.0,13.0,2.67,61.54,0.0,0.0,...,3.0,9.50,57.0,47.0,3.0,15.67,4.95,19.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,Yuvraj Singh,4.0,4.0,0.0,52.0,52.0,13.0,100.0,0.0,0.0,...,1.0,3.50,21.0,19.0,1.0,19.0,5.43,21.0,0.0,0.0
486,ZE Green,10.0,10.0,0.0,110.0,134.0,11.0,82.09,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
487,Zahoor Khan,1.0,1.0,1.0,1.0,1.0,None,100.0,0.0,0.0,...,3.0,12.67,76.0,59.0,5.0,11.8,4.66,15.2,0.0,0.0
488,Zawar Farid,1.0,1.0,0.0,2.0,4.0,2.0,50.0,0.0,0.0,...,1.0,2.83,17.0,25.0,0.0,None,8.82,None,0.0,0.0


In [ ]:
## CricBuzz Commentary Fetch – All 44 T20 WC 2024 Matches

Fetches ball-by-ball commentary from CricBuzz for all 44 ICC Men's T20 World Cup 2024 matches using CricBuzz's internal API (`/api/mcenter/{matchId}/full-commentary/{inningId}`).

**Approach:**
1. Probe CricBuzz IDs 87480–87885 to find which IDs correspond to T20 WC 2024 (series 7476).
2. Match those CricBuzz IDs to Cricsheet IDs via team names + date.
3. Fetch full ball-by-ball commentary for both innings.
4. Parse commentary (over, ball, text) and merge into `master_deliveries_with_commentary.csv`.

**Result:** 9,693 delivery rows across all 44 matches now have ball commentary.

In [ ]:
import re, time, json as _json
import requests as _req

# ── Probe CricBuzz IDs to find all T20 WC 2024 matches ───────────────────────
# CricBuzz internal API (no key required):
#   GET /api/mcenter/{matchId}/full-commentary/{inningId}
# Returns JSON with matchDetails.matchHeader containing seriesId, team names, date.
# T20 WC 2024 series ID = 7476; match IDs span roughly 87480–87885.

_HDR = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.cricbuzz.com/",
}

_TEAM_NORM = {
    "india": "india", "ireland": "ireland", "pakistan": "pakistan",
    "australia": "australia", "england": "england", "bangladesh": "bangladesh",
    "sri lanka": "sri lanka", "south africa": "south africa",
    "new zealand": "new zealand", "west indies": "west indies",
    "afghanistan": "afghanistan", "nepal": "nepal", "netherlands": "netherlands",
    "scotland": "scotland", "canada": "canada", "uganda": "uganda",
    "papua new guinea": "papua new guinea", "namibia": "namibia",
    "oman": "oman", "united states of america": "usa", "usa": "usa",
    "united states": "usa",
}

def _norm(n):
    return _TEAM_NORM.get(str(n).strip().lower(), str(n).strip().lower())

def probe_cb_match(cb_id):
    """Return {cb_id, t1, t2, date} if this is a T20 WC 2024 match, else None."""
    url = f"https://www.cricbuzz.com/api/mcenter/{cb_id}/full-commentary/1"
    try:
        r = _req.get(url, headers=_HDR, timeout=10)
        if r.status_code != 200:
            return None
        mh = r.json().get("matchDetails", {}).get("matchHeader", {})
        if mh.get("seriesId") != 7476:
            return None
        t1 = _norm(mh.get("team1", {}).get("name", ""))
        t2 = _norm(mh.get("team2", {}).get("name", ""))
        date_ms = mh.get("matchStartTimestamp", 0)
        date = pd.to_datetime(int(date_ms), unit="ms").date() if date_ms else None
        return {"cb_id": cb_id, "t1": t1, "t2": t2, "date": str(date)}
    except Exception:
        return None

# Load or build scan cache
_scan_cache = os.path.join(OUTPUT_DIR, "raw_commentary_cb", "_scan_results.json")
os.makedirs(os.path.dirname(_scan_cache), exist_ok=True)

if os.path.exists(_scan_cache):
    with open(_scan_cache) as _f:
        _cb_matches = _json.load(_f)
    print(f"Loaded {len(_cb_matches)} T20WC24 matches from cache")
else:
    print("Scanning CricBuzz IDs 87480-87885 for T20 WC 2024 matches ...")
    _cb_matches = []
    for _id in range(87480, 87886):
        _info = probe_cb_match(_id)
        if _info:
            print(f"  Found {_id}: {_info['t1']} vs {_info['t2']} | {_info['date']}")
            _cb_matches.append(_info)
        time.sleep(0.3)
    with open(_scan_cache, "w") as _f:
        _json.dump(_cb_matches, _f)
    print(f"Scan complete: {len(_cb_matches)} T20WC24 matches found")

# Build Cricsheet -> CricBuzz mapping
_df_m = pd.read_csv(os.path.join(OUTPUT_DIR, "master_matches.csv"))
_df_m["date"] = pd.to_datetime(_df_m["date"], errors="coerce")
_df_2024 = _df_m[_df_m["date"].dt.year == 2024].copy()

CRICSHEET_TO_CRICBUZZ = {}
_unmatched = []
for _, row in _df_2024.iterrows():
    cs_id = str(row["match_id"])
    ct1, ct2 = _norm(row["team1"]), _norm(row["team2"])
    cs_teams = frozenset([ct1, ct2])
    cs_date = row["date"].date()
    matched = False
    for cb in _cb_matches:
        if frozenset([cb["t1"], cb["t2"]]) == cs_teams:
            cb_date = pd.to_datetime(cb["date"]).date() if cb["date"] and cb["date"] != "None" else None
            if cb_date and abs((cs_date - cb_date).days) <= 2:
                CRICSHEET_TO_CRICBUZZ[cs_id] = cb["cb_id"]
                matched = True
                break
    if not matched:
        _unmatched.append((cs_id, row["team1"], row["team2"]))

print(f"Mapped: {len(CRICSHEET_TO_CRICBUZZ)} / 44")
if _unmatched:
    print("Unmatched:", _unmatched)
else:
    print("All 44 matches mapped.")

In [ ]:
# ── PASTE YOUR RAPIDAPI KEY HERE ─────────────────────────────────────────────
# Fetch ball-by-ball commentary using CricBuzz internal API (no API key required)
# Endpoint: GET /api/mcenter/{matchId}/full-commentary/{inningId}
# inningId: 1 = first innings, 2 = second innings

RAW_COMM_DIR = os.path.join(OUTPUT_DIR, "raw_commentary_cb")
os.makedirs(RAW_COMM_DIR, exist_ok=True)

def parse_commentary_list_cb(commentary_list, inning_no):
    """Parse a CricBuzz commentaryList into structured rows."""
    rows = []
    for entry in commentary_list:
        ball_nbr = entry.get("ballNbr", 0)
        event    = entry.get("event", "NONE")
        if ball_nbr == 0 and event in ("NONE", None):
            continue
        over_number = entry.get("overNumber")
        if over_number is not None:
            try:
                ov_f = float(over_number)
                ov   = int(ov_f)
                bl   = round((ov_f - ov) * 10)
            except Exception:
                continue
        elif ball_nbr > 0:
            ov = (ball_nbr - 1) // 6
            bl = ((ball_nbr - 1) % 6) + 1
        else:
            continue
        if bl < 1:
            continue
        comm_text = re.sub(r'B\d+\$\s*', '', entry.get("commText", "") or "").strip()
        if not comm_text:
            continue
        rows.append({
            "inning": inning_no, "over": ov, "ball": bl,
            "short_text": comm_text[:120], "full_text": comm_text,
        })
    return rows


print(f"Mapped matches: {len(CRICSHEET_TO_CRICBUZZ)} / 44")
print("Fetching commentary ...")

all_commentary = {}
for cs_id, cb_id in CRICSHEET_TO_CRICBUZZ.items():
    cache_path = os.path.join(RAW_COMM_DIR, f"{cs_id}_rows.json")
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            rows = json.load(f)
        if rows:
            all_commentary[cs_id] = rows
            continue

    rows = []
    for inning in [1, 2]:
        url = f"https://www.cricbuzz.com/api/mcenter/{cb_id}/full-commentary/{inning}"
        try:
            r = requests.get(url, headers={"User-Agent": "Mozilla/5.0",
                                            "Accept": "application/json"},
                             timeout=20)
            if r.status_code != 200:
                continue
            data = r.json()
        except Exception as e:
            print(f"  [{cs_id}] inning {inning}: {e}")
            continue
        for inn_obj in data.get("commentary", []):
            inn_id  = inn_obj.get("inningsId", inning)
            inn_no  = 1 if inn_id <= 1 else 2
            parsed  = parse_commentary_list_cb(inn_obj.get("commentaryList", []), inn_no)
            rows.extend(parsed)
        time.sleep(0.5)

    if rows:
        with open(cache_path, "w") as f:
            json.dump(rows, f)
        all_commentary[cs_id] = rows
    time.sleep(0.8)

print(f"Matches with commentary: {len(all_commentary)}")
print(f"Total rows: {sum(len(v) for v in all_commentary.values())}")
# ─────────────────────────────────────────────────────────────────────────────

RAPIDAPI_HOST = "cricbuzz-cricket.p.rapidapi.com"
RAPIDAPI_HDR = {
    "x-rapidapi-key": RAPIDAPI_KEY,
    "x-rapidapi-host": RAPIDAPI_HOST,
}
BASE_URL = f"https://{RAPIDAPI_HOST}"

RAW_COMM_DIR = os.path.join(OUTPUT_DIR, "raw_commentary")
os.makedirs(RAW_COMM_DIR, exist_ok=True)

def fetch_commentary_for_match(cb_match_id, cricsheet_id):
    """
    Fetch all ball-by-ball commentary for both innings of a match.
    Saves raw JSON to raw_commentary/{cricsheet_id}.json.
    Returns list of raw commentary event dicts.
    """
    save_path = os.path.join(RAW_COMM_DIR, f"{cricsheet_id}.json")
    # Skip if already fetched
    if os.path.exists(save_path):
        with open(save_path, "r", encoding="utf-8") as f:
            return json.load(f)

    all_events = []
    # CricBuzz paginates: fetch inning 1 then inning 2; each can have multiple pages
    for innings_id in [1, 2]:
        page = 0
        while True:
            params = {"InningsId": innings_id}
            if page > 0:
                # Use the over from the last event fetched to paginate backwards
                if all_events:
                    last_over = min(
                        e.get("overNumber", 0) for e in all_events
                        if e.get("inningsId") == innings_id
                    ) if any(e.get("inningsId") == innings_id for e in all_events) else 0
                    params["overNumber"] = last_over
            url = f"{BASE_URL}/mcenter/v1/{cb_match_id}/comm"
            try:
                resp = requests.get(url, headers=RAPIDAPI_HDR, params=params, timeout=20)
                resp.raise_for_status()
                data = resp.json()
            except Exception as e:
                print(f"  Error fetching match {cb_match_id} innings {innings_id} page {page}: {e}")
                break
            events = data.get("commentaryList", [])
            if not events:
                break
            all_events.extend(events)
            # Check if more pages remain for this innings
            if not data.get("lastFetchedInningNumber") or not events:
                break
            # Stop if we have fetched all overs (over 0 = last page going backwards)
            min_over = min(e.get("overNumber", 999) for e in events)
            if min_over <= 0:
                break
            page += 1
            time.sleep(0.5)  # be polite

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(all_events, f, ensure_ascii=False, indent=2)
    return all_events


# ── Fetch all mapped matches ──────────────────────────────────────────────────
print(f"Fetching commentary for {len(CRICSHEET_TO_CRICBUZZ)} matches …")
print(f"Raw JSON saved in: {RAW_COMM_DIR}\n")

_fetch_errors = []
for _cs_id, _cb_id in CRICSHEET_TO_CRICBUZZ.items():
    _save = os.path.join(RAW_COMM_DIR, f"{_cs_id}.json")
    if os.path.exists(_save):
        print(f"  [{_cs_id}] Already fetched – skipping")
        continue
    print(f"  [{_cs_id}] Fetching CricBuzz match {_cb_id} …", end=" ")
    try:
        evts = fetch_commentary_for_match(_cb_id, _cs_id)
        print(f"{len(evts)} events")
    except Exception as e:
        print(f"ERROR: {e}")
        _fetch_errors.append(_cs_id)
    time.sleep(1)  # rate-limit courtesy pause

print(f"\nDone. Errors: {_fetch_errors if _fetch_errors else 'None'}")
print(f"Files in raw_commentary/: {len(os.listdir(RAW_COMM_DIR))}")

In [ ]:
# Merge all commentary into master_deliveries_with_commentary.csv
df_del = pd.read_csv(os.path.join(OUTPUT_DIR, "master_deliveries.csv"))
df_del["match_id"] = df_del["match_id"].astype(str)

comm_rows = []
for cs_id, rows in all_commentary.items():
    for r in rows:
        comm_rows.append({
            "match_id": cs_id, "inning_no": r["inning"],
            "over": r["over"], "ball_in_over": r["ball"],
            "commentary_short": r["short_text"], "commentary": r["full_text"],
        })
df_comm = pd.DataFrame(comm_rows)

df_del["commentary_short"] = None
df_del["commentary"] = None
for (mid, inn), g_d in df_del.groupby(["match_id", "inning_no"]):
    g_d_s = g_d.sort_values(["over", "ball_in_over"])
    g_c = df_comm[(df_comm["match_id"] == mid) & (df_comm["inning_no"] == inn)].sort_values(["over", "ball_in_over"])
    if g_c.empty:
        continue
    idx_d, idx_c = g_d_s.index.tolist(), g_c.index.tolist()
    for i, didx in enumerate(idx_d):
        if i < len(idx_c):
            df_del.at[didx, "commentary_short"] = g_c.at[idx_c[i], "commentary_short"]
            df_del.at[didx, "commentary"]       = g_c.at[idx_c[i], "commentary"]

out_path = os.path.join(OUTPUT_DIR, "master_deliveries_with_commentary.csv")
df_del.to_csv(out_path, index=False)

filled  = df_del["commentary"].notna().sum()
matches = df_del[df_del["commentary"].notna()]["match_id"].nunique()
print(f"Saved: {out_path}")
print(f"Commentary: {filled}/{len(df_del)} rows ({filled/len(df_del)*100:.1f}%) across {matches} matches")

# Preview
df_del[df_del["commentary"].notna()][
    ["match_id","inning_no","over","ball_in_over","batter","bowler","runs_total","commentary"]
].head(5)